In [1]:
import json
import re
import random
import os
from datasets import load_dataset
from tqdm import tqdm

from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
DATASET_NAME = "glaiveai/glaive-function-calling-v2"
OUTPUT_DIR = "data"

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

MIN_EXAMPLES = 2000
TARGET_TOTAL = 5000
RANDOM_SEED = 42

In [4]:
INSTRUCTIONS = [
    "Extract the key information from the following text and return it as JSON.",
    "Parse the input text and output structured JSON with entities and values.",
    "Extract all explicit entities and values as JSON.",
    "Convert the text into structured JSON format.",
    "Identify and extract relevant structured information from the input text.",
    "Return only JSON containing extracted fields."
]

In [5]:
def extract_user_message(chat_text):
    """Extract the first USER message from chat text."""
    pattern = r'USER:\s*(.*?)(?:\n\n|ASSISTANT:|$)'
    match = re.search(pattern, chat_text, re.DOTALL)
    return match.group(1).strip() if match else None

In [6]:
def extract_functioncall_json(chat_text):
    """Extract the JSON from <functioncall> blocks in chat text."""
    pattern = r'<functioncall>\s*({.*?})(?:\s*<\|endoftext\|>|$)'
    matches = re.findall(pattern, chat_text, re.DOTALL)

    results = []
    for m in matches:
        try:
            results.append(json.loads(m))
        except:
            pass
    return results

In [7]:
def clean_extraction_json(parsed_json):
    """
    Clean the JSON for true extraction format.
    - REMOVE 'name' key (function name) → we don't need it
    - FLATTEN 'arguments' up one level → pure entities
    - Return {} if no meaningful data
    
    BUG FIX: Check if 'arguments' is a dict before updating
    """
    if not isinstance(parsed_json, dict):
        return {}  # Return {} not None

    result = {}

    # FIXED: Check if arguments exists AND is a dict
    if "arguments" in parsed_json and isinstance(parsed_json["arguments"], dict):
        result.update(parsed_json["arguments"])

    # Merge any top-level keys (except 'name' and 'arguments')
    for k, v in parsed_json.items():
        if k not in ["name", "arguments"]:
            result[k] = v

    return result

In [8]:
def validate_example(ex):
    """Validate that example is clean and valid."""
    if not ex:
        return False
    if not ex["input"] or len(ex["input"]) < 5:
        return False

    try:
        json.loads(ex["output"])
    except:
        return False

    return True

In [9]:
def format_for_training(instruction, user_input, extracted_json):
    """Format into the instruction-input-output text format for training."""
    return {
        "instruction": instruction,
        "input": user_input,
        "output": json.dumps(extracted_json, ensure_ascii=False),
        "text": f"""### Instruction:
{instruction}

### Input:
{user_input}

### Response:
{json.dumps(extracted_json, ensure_ascii=False)}"""
    }

In [10]:
def convert_example(row):
    """
    Convert one raw glaive row to JSON extraction format.
    """
    chat = row.get("chat", "")

    # Get the user's natural language question
    user_msg = extract_user_message(chat)
    if not user_msg:
        return None

    # Extract the function call JSON (contains arguments with entities)
    fc_jsons = extract_functioncall_json(chat)
    if not fc_jsons:
        return None

    # Clean it: remove function name, keep only entities
    cleaned = clean_extraction_json(fc_jsons[0])

    # Pick a random instruction template
    instruction = random.choice(INSTRUCTIONS)

    # Format for training
    return format_for_training(instruction, user_msg, cleaned)

In [11]:
print("Loading dataset...")

ds = load_dataset(DATASET_NAME, split="train")

print(f"✓ Loaded: {len(ds)} examples")

Loading dataset...
✓ Loaded: 112960 examples


In [12]:
print(ds)
print(type(ds))
print()
print("Sample:")
print("System:", ds[0]['system'][:200])
print()
print("Chat:", ds[0]['chat'][:300])

Dataset({
    features: ['system', 'chat'],
    num_rows: 112960
})
<class 'datasets.arrow_dataset.Dataset'>

Sample:
System: SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "get_exchange_rate",
    "description": "Get the exchange rate between two currencies",

Chat: USER: Can you book a flight for me from New York to London?


ASSISTANT: I'm sorry, but I don't have the capability to book flights. My current function allows me to get the exchange rate between two currencies. If you need help with that, feel free to ask! <|endoftext|>





In [13]:
# Debug: Check a few conversions before processing all
print("Testing conversion on first 5 examples...\n")
for i in range(5):
    ex = convert_example(ds[i])
    if ex:
        print(f"Example {i}:")
        print(f"  Input: {ex['input'][:80]}")
        print(f"  Output: {ex['output'][:80]}")
        print()
    else:
        print(f"Example {i}: Skipped (None returned)\n")

Testing conversion on first 5 examples...

Example 0: Skipped (None returned)

Example 1: Skipped (None returned)

Example 2: Skipped (None returned)

Example 3: Skipped (None returned)

Example 4: Skipped (None returned)



In [14]:
print("Converting dataset to JSON extraction format...\n")

converted = []
skipped = 0

# Option 1: Use tqdm directly on dataset (no need to convert to list first)
for row in tqdm(ds, total=min(len(ds), TARGET_TOTAL * 2)):
    try:
        ex = convert_example(row)

        if validate_example(ex):
            converted.append(ex)
        else:
            skipped += 1
    except Exception as e:
        print(f"Error processing row: {e}")
        skipped += 1

    if len(converted) >= TARGET_TOTAL:
        break

print(f"\n✓ Valid: {len(converted)}")
print(f"✗ Skipped: {skipped}")

Converting dataset to JSON extraction format...



112960it [00:02, 47295.66it/s]                         


✓ Valid: 2378
✗ Skipped: 110582


In [15]:
print("Shuffling and splitting data...\n")

random.seed(RANDOM_SEED)
random.shuffle(converted)

n = len(converted)

n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

train = converted[:n_train]
val = converted[n_train:n_train+n_val]
test = converted[n_train+n_val:]

print(f"Train: {len(train)}")
print(f"Val: {len(val)}")
print(f"Test: {len(test)}")

Shuffling and splitting data...

Train: 1902
Val: 237
Test: 239


In [16]:
BASE_DIR = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, "data")

print("Saving inside:", OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

def save(name, data):
    path = os.path.join(OUTPUT_DIR, f"{name}.jsonl")
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"✓ {path} : {len(data)} examples")

save("train", train)
save("val", val)
save("test", test)

Saving inside: c:\Users\ganes_3ck5\DataScience\Gen_AI\Course_GenAI\Gen_AI_In-Depth\resume_projects\llm-finetune\notebooks\data
✓ c:\Users\ganes_3ck5\DataScience\Gen_AI\Course_GenAI\Gen_AI_In-Depth\resume_projects\llm-finetune\notebooks\data\train.jsonl : 1902 examples
✓ c:\Users\ganes_3ck5\DataScience\Gen_AI\Course_GenAI\Gen_AI_In-Depth\resume_projects\llm-finetune\notebooks\data\val.jsonl : 237 examples
✓ c:\Users\ganes_3ck5\DataScience\Gen_AI\Course_GenAI\Gen_AI_In-Depth\resume_projects\llm-finetune\notebooks\data\test.jsonl : 239 examples


In [17]:
print("\n" + "="*80)
print("DATA PREPARATION COMPLETE ✅")
print("="*80)
print(f"Train: {len(train)}")
print(f"Val:   {len(val)}")
print(f"Test:  {len(test)}")
print(f"Total: {len(converted)}")
print("="*80)
print("\nNext steps:")
print("1. Run check_data.py to validate quality")
print("2. Upload data/ folder to Google Drive")
print("3. Run SFT training in Colab")
print("="*80)


DATA PREPARATION COMPLETE ✅
Train: 1902
Val:   237
Test:  239
Total: 2378

Next steps:
1. Run check_data.py to validate quality
2. Upload data/ folder to Google Drive
3. Run SFT training in Colab


In [18]:
# Quick validation: check a few examples
print("\nValidation Sample:")
print("="*80)
for i in range(min(3, len(train))):
    ex = train[i]
    print(f"\nExample {i+1}:")
    print(f"Instruction: {ex['instruction']}")
    print(f"Input:       {ex['input'][:60]}...")
    print(f"Output:      {ex['output']}")
    # Verify output is valid JSON
    try:
        json.loads(ex['output'])
        print("Status:      ✓ Valid JSON")
    except:
        print("Status:      ✗ INVALID JSON")
print("\n" + "="*80)


Validation Sample:

Example 1:
Instruction: Convert the text into structured JSON format.
Input:       Hi, can you help me with something?...
Output:      {}
Status:      ✓ Valid JSON

Example 2:
Instruction: Convert the text into structured JSON format.
Input:       Hey, I'm feeling a bit bored. Can you tell me something inte...
Output:      {}
Status:      ✓ Valid JSON

Example 3:
Instruction: Identify and extract relevant structured information from the input text.
Input:       Hi, I would like to calculate my BMI. I weigh 70 kilograms a...
Output:      {"weight": 70, "height": 1.75}
Status:      ✓ Valid JSON



In [19]:
import json

empty_count = 0
real_count = 0
invalid_count = 0

for ex in train:
    try:
        obj = json.loads(ex["output"])

        if isinstance(obj, dict) and len(obj) == 0:
            empty_count += 1
        else:
            real_count += 1

    except:
        invalid_count += 1

print("Total examples:", len(train))
print("Real JSON (with data):", real_count)
print("Empty JSON {}:", empty_count)
print("Invalid JSON:", invalid_count)

Total examples: 1902
Real JSON (with data): 146
Empty JSON {}: 1756
Invalid JSON: 0


In [20]:
import json
import os

def split_real_empty(dataset):
    real, empty, invalid = [], [], []

    for ex in dataset:
        try:
            obj = json.loads(ex["output"])

            if isinstance(obj, dict):
                if len(obj) == 0:
                    empty.append(ex)
                else:
                    real.append(ex)
            else:
                invalid.append(ex)

        except:
            invalid.append(ex)

    return real, empty, invalid


def save_jsonl(path, data):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


# -----------------------------
# OUTPUT DIRECTORY
# -----------------------------
out_dir = "data_split"
os.makedirs(out_dir, exist_ok=True)


# -----------------------------
# SPLIT TRAIN / VAL / TEST
# -----------------------------
train_real, train_empty, train_invalid = split_real_empty(train)
val_real, val_empty, val_invalid = split_real_empty(val)
test_real, test_empty, test_invalid = split_real_empty(test)


# -----------------------------
# PRINT STATS
# -----------------------------
print("\nTRAIN:")
print("Real:", len(train_real), "Empty:", len(train_empty), "Invalid:", len(train_invalid))

print("\nVAL:")
print("Real:", len(val_real), "Empty:", len(val_empty), "Invalid:", len(val_invalid))

print("\nTEST:")
print("Real:", len(test_real), "Empty:", len(test_empty), "Invalid:", len(test_invalid))


# -----------------------------
# SAVE FILES
# -----------------------------
# TRAIN
save_jsonl(os.path.join(out_dir, "train_real.jsonl"), train_real)
save_jsonl(os.path.join(out_dir, "train_empty.jsonl"), train_empty)

# VAL
save_jsonl(os.path.join(out_dir, "val_real.jsonl"), val_real)
save_jsonl(os.path.join(out_dir, "val_empty.jsonl"), val_empty)

# TEST
save_jsonl(os.path.join(out_dir, "test_real.jsonl"), test_real)
save_jsonl(os.path.join(out_dir, "test_empty.jsonl"), test_empty)

print("\n✔ All datasets saved successfully in:", out_dir)


TRAIN:
Real: 146 Empty: 1756 Invalid: 0

VAL:
Real: 21 Empty: 216 Invalid: 0

TEST:
Real: 20 Empty: 219 Invalid: 0

✔ All datasets saved successfully in: data_split
